# MSAlign raw-input playground

This notebook scores candidate molecules directly from a centroided spectrum. DreaMS, MolDeBERTa, spectral bins, and Morgan fingerprints are computed on demand; no dataset preprocessing is required.

In VS Code, select `playground/.venv/bin/python` as the notebook kernel, then run the cells normally.

In [ ]:
spectrum = {
    # Centroided [m/z, intensity] peaks. Intensities need not be normalized.
    'peaks': [
        [43.0184, 0.18],
        [77.0390, 0.12],
        [105.0335, 0.46],
        [121.0284, 1.00],
        [138.0550, 0.31],
    ],
    'precursor_mz': 181.0495,
    # Both fields below are optional and may also be None.
    'adduct': '[M+H]+',
    'collision_energy': 30.0,
}

candidate = 'CC(=O)Oc1ccccc1C(=O)O'
candidates = [candidate, 'CCO', 'CCCO', 'c1ccccc1']

## What should already be preprocessed?

Provide a centroided MS/MS peak list, sorted by increasing `m/z`. Centroiding, deisotoping, noise removal, duplicate-peak merging, salt removal, and parent-molecule selection are not performed here. Intensities need not be normalized. `precursor_mz` is the measured ion m/z; do not insert it into `peaks`, because the DreaMS transform adds its own precursor pseudo-peak. Candidate strings must be valid RDKit SMILES for neutral molecules.

Inside `MSAlign`, DreaMS keeps the 100 most intense peaks, normalizes their intensities, pads the peak list, and prepends the precursor. Fused MSAlign+ also creates 0.1-Th spectrum bins, radius-2 4096-bit Morgan fingerprints, candidate exact masses, and the calibrated score combination. MolDeBERTa and DreaMS embeddings are computed on demand. Missing metadata uses learned missing/unknown encodings; a missing or unsupported adduct makes the MSAlign+ mass channels zero. See [`playground/README.md`](README.md) for the complete input contract.

In [ ]:
from pathlib import Path
import sys

# VS Code may start the kernel either here or in the repository root.
repository_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / 'playground' / '__init__.py').is_file()),
    None,
)
if repository_root is None:
    raise RuntimeError('Open this notebook from inside the MSAlign repository')
sys.path.insert(0, str(repository_root))

from playground import MSAlign

# Replace this path with an MSAlign+ bundle to use score fusion.
checkpoint = repository_root / 'checkpoints' / 'msalign_massspecgym_formula_seed0'
model = MSAlign(checkpoint)

In [ ]:
score = model(spectrum, candidate)
score

In [ ]:
# The returned scores follow the input SMILES order.
scores = model(spectrum, candidates)
list(zip(candidates, scores))